# Ad Click Prediction Case Study ? Corrected Version

This notebook implements the reviewer feedback: leakage-safe historical CTR features, explicit baselines, feature ablation, corrected model-selection rationale, cautious performance language, individual feature importance, and clearer business mapping.

## 1. Executive Summary

The solution predicts rare ad clicks using temporal, contextual, and historical CTR features. Because validation performance is moderate rather than high, predictions should be used as a ranking and targeting aid, with threshold tuning and live monitoring.

## 2. Business Problem and ML Objective

The ad-tech objective is to reduce wasted impressions and prioritize users with higher click propensity. The ML task is imbalanced binary classification, evaluated with ROC-AUC, PR-AUC, F1, precision, recall, and confusion-matrix trade-offs.

## 3. Dataset Understanding

The training data contains impression-level records with user, product, campaign, webpage, demographic, and timestamp fields. The target is `is_click`.

## 4. Validation Strategy

A random stratified split is not used as the primary validation strategy because ad serving is temporal. The middle date is used for threshold tuning and latest date is held out to approximate future serving and avoid future behavior leaking into model development.

## 5. Leakage-Safe Historical Feature Engineering

Historical CTR features are computed cumulatively for training rows. Each row only uses impressions that occurred before the current impression. Threshold rows use training-history maps; final holdout rows use all pre-holdout history; test rows use all training-history maps.

## 6. Baselines, Ablation, Models, and Evaluation

The notebook compares always-negative and global-CTR baselines, feature-set ablations, class-weighted models, threshold tuning, ROC/PR curves, calibration, and feature importance.

## 7. Business Questions and Recommendations

The analysis connects model evidence to weekend bidding, product performance, personalization, SMOTE trade-offs, inventory planning signals, user profiles, and operational monitoring.

In [1]:

# Ad Click Prediction - corrected leakage-safe case study
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, precision_recall_curve, roc_curve
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from imblearn.over_sampling import SMOTENC
import time

RANDOM_STATE = 42
DATA_DIR = Path('assets/data')
IMAGE_DIR = Path('assets/images')
OUTPUT_DIR = Path('outputs')
REPORT_DIR = Path('reports')
for d in [IMAGE_DIR, OUTPUT_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'Ad_click_prediction_train (1).csv'
TEST_PATH = DATA_DIR / 'Ad_Click_prediciton_test.csv'

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
train['DateTime'] = pd.to_datetime(train['DateTime'])
test['DateTime'] = pd.to_datetime(test['DateTime'])
print('Train:', train.shape, 'Test:', test.shape)
print('Click rate:', round(train['is_click'].mean() * 100, 2), '%')

def add_time_features(df):
    out = df.copy()
    out['hour'] = out['DateTime'].dt.hour
    out['day'] = out['DateTime'].dt.day
    out['dayofweek'] = out['DateTime'].dt.dayofweek
    out['is_weekend'] = (out['dayofweek'] >= 5).astype(int)
    out['is_night'] = out['hour'].between(0, 5).astype(int)
    out['is_business_hour'] = out['hour'].between(9, 18).astype(int)
    return out

def add_interaction_keys(df):
    out = df.copy()
    out['user_product'] = out['user_id'].astype(str) + '_' + out['product'].astype(str)
    out['campaign_product'] = out['campaign_id'].astype(str) + '_' + out['product'].astype(str)
    out['webpage_product'] = out['webpage_id'].astype(str) + '_' + out['product'].astype(str)
    return out

def add_cumulative_history_features(df, cols, target='is_click', smoothing=30, initial_prior=0.05):
    """Leakage-safe historical CTR/counts for training rows.
    Each row only sees impressions/clicks that occurred before itself.
    The smoothing prior is also expanding, so future clicks are not used.
    """
    out = df.sort_values(['DateTime', 'session_id']).copy()
    global_previous_count = np.arange(len(out), dtype=float)
    global_previous_clicks = out[target].cumsum().shift(fill_value=0).astype(float)
    expanding_prior = (global_previous_clicks + smoothing * initial_prior) / (global_previous_count + smoothing)
    for col in cols:
        previous_count = out.groupby(col).cumcount()
        previous_clicks = out.groupby(col)[target].cumsum() - out[target]
        out[f'{col}_hist_impressions'] = previous_count.astype(float)
        out[f'{col}_hist_clicks'] = previous_clicks.astype(float)
        out[f'{col}_hist_ctr'] = (previous_clicks + smoothing * expanding_prior) / (previous_count + smoothing)
    final_prior = float((out[target].sum() + smoothing * initial_prior) / (len(out) + smoothing))
    return out.sort_index(), final_prior

def build_history_maps(df, cols, target='is_click', smoothing=30):
    prior = df[target].mean()
    maps = {}
    for col in cols:
        stats = df.groupby(col)[target].agg(['sum', 'count'])
        stats['hist_ctr'] = (stats['sum'] + smoothing * prior) / (stats['count'] + smoothing)
        maps[col] = {
            'ctr': stats['hist_ctr'].to_dict(),
            'clicks': stats['sum'].to_dict(),
            'impressions': stats['count'].to_dict(),
        }
    return maps, prior

def apply_history_maps(df, maps, prior, cols):
    out = df.copy()
    for col in cols:
        out[f'{col}_hist_impressions'] = out[col].map(maps[col]['impressions']).fillna(0).astype(float)
        out[f'{col}_hist_clicks'] = out[col].map(maps[col]['clicks']).fillna(0).astype(float)
        out[f'{col}_hist_ctr'] = out[col].map(maps[col]['ctr']).fillna(prior).astype(float)
    return out

train_fe = add_interaction_keys(add_time_features(train))
test_fe = add_interaction_keys(add_time_features(test))
holdout_cutoff = train_fe['DateTime'].max().normalize()
threshold_cutoff = holdout_cutoff - pd.Timedelta(days=1)
train_raw = train_fe[train_fe['DateTime'] < threshold_cutoff].copy()
threshold_raw = train_fe[(train_fe['DateTime'] >= threshold_cutoff) & (train_fe['DateTime'] < holdout_cutoff)].copy()
holdout_raw = train_fe[train_fe['DateTime'] >= holdout_cutoff].copy()
print('Training period ends before:', threshold_cutoff.date())
print('Threshold-tuning date:', threshold_cutoff.date())
print('Final holdout date:', holdout_cutoff.date())
print('Train:', train_raw.shape, 'Threshold:', threshold_raw.shape, 'Holdout:', holdout_raw.shape, 'Holdout click rate:', round(holdout_raw.is_click.mean(), 4))
print('Random stratified split is not primary because future ad-serving behavior should not leak backward into training.')

history_cols = [
    'product', 'campaign_id', 'webpage_id', 'product_category_1', 'product_category_2',
    'user_group_id', 'age_level', 'city_development_index', 'user_product',
    'campaign_product', 'webpage_product'
]
interaction_cols = ['user_product', 'campaign_product', 'webpage_product']

train_hist, train_prior = add_cumulative_history_features(train_raw, history_cols)
train_maps, _ = build_history_maps(train_raw, history_cols)
threshold_hist = apply_history_maps(threshold_raw, train_maps, train_prior, history_cols)
pre_holdout_history = pd.concat([train_raw, threshold_raw], axis=0)
holdout_maps, holdout_prior = build_history_maps(pre_holdout_history, history_cols)
holdout_hist = apply_history_maps(holdout_raw, holdout_maps, holdout_prior, history_cols)

# Keep model iteration practical while preserving untouched threshold and holdout periods.
model_train_idx, _ = train_test_split(
    np.arange(len(train_hist)),
    train_size=min(150000, len(train_hist)),
    stratify=train_hist['is_click'],
    random_state=RANDOM_STATE,
)
model_train = train_hist.iloc[model_train_idx].copy()
print('Model training sample:', model_train.shape, 'Threshold tuning:', threshold_hist.shape, 'Final holdout:', holdout_hist.shape)

baseline_features = [
    'product','gender','campaign_id','webpage_id','product_category_1','product_category_2',
    'user_group_id','age_level','user_depth','city_development_index','var_1',
    'hour','dayofweek','is_weekend','is_night','is_business_hour'
]
interaction_count_features = [f'{c}_hist_impressions' for c in interaction_cols] + [f'{c}_hist_clicks' for c in interaction_cols]
historical_ctr_features = [f'{c}_hist_ctr' for c in history_cols]
all_numeric_features = interaction_count_features + historical_ctr_features
full_features = baseline_features + all_numeric_features
cat_features = baseline_features

def make_preprocess(numeric_features):
    return ColumnTransformer([
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=20)),
        ]), cat_features),
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_features),
    ])

def evaluate_probs(y_true, probs, name, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    return {
        'model': name, 'threshold': float(threshold),
        'roc_auc': roc_auc_score(y_true, probs),
        'pr_auc': average_precision_score(y_true, probs),
        'f1': f1_score(y_true, preds, zero_division=0),
        'precision': precision_score(y_true, preds, zero_division=0),
        'recall': recall_score(y_true, preds, zero_division=0),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)
    }

def best_threshold_by_f1(y_true, probs):
    thresholds = np.unique(np.round(np.linspace(0.01, 0.99, 199), 3))
    f1s = [f1_score(y_true, probs >= t, zero_division=0) for t in thresholds]
    return float(thresholds[int(np.argmax(f1s))])

def fit_eval_model(name, estimator, features, numeric_features):
    pipe = Pipeline([('prep', make_preprocess(numeric_features)), ('model', estimator)])
    pipe.fit(model_train[features], model_train['is_click'])
    threshold_probs = pipe.predict_proba(threshold_hist[features])[:, 1]
    threshold = best_threshold_by_f1(threshold_hist['is_click'], threshold_probs)
    holdout_probs = pipe.predict_proba(holdout_hist[features])[:, 1]
    return pipe, holdout_probs, [evaluate_probs(holdout_hist['is_click'], holdout_probs, name, 0.5), evaluate_probs(holdout_hist['is_click'], holdout_probs, name + ' - tuned threshold', threshold)]

# Proper baselines
baseline_rows = []
y_valid = holdout_hist['is_click']
global_ctr = model_train['is_click'].mean()
baseline_rows.append(evaluate_probs(y_valid, np.full(len(y_valid), global_ctr), 'Global CTR probability baseline', global_ctr))
baseline_rows.append({
    'model': 'Always non-click baseline', 'threshold': 0.5, 'roc_auc': 0.5,
    'pr_auc': y_valid.mean(), 'f1': 0.0, 'precision': 0.0, 'recall': 0.0,
    'tn': int((y_valid == 0).sum()), 'fp': 0, 'fn': int((y_valid == 1).sum()), 'tp': 0
})

# Feature ablation using the same balanced logistic estimator for clean comparison.
ablation_specs = [
    ('Baseline raw + temporal', baseline_features, []),
    ('+ Interaction history counts', baseline_features + interaction_count_features, interaction_count_features),
    ('+ Leakage-safe historical CTR', full_features, all_numeric_features),
]
ablation_rows = []
for label, feats, nums in ablation_specs:
    estimator = LogisticRegression(max_iter=500, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE)
    _, probs, rows = fit_eval_model(label, estimator, feats, nums)
    tuned = [r for r in rows if r['model'].endswith('tuned threshold')][0]
    ablation_rows.append(tuned)
ablation = pd.DataFrame(ablation_rows)
display(pd.DataFrame(baseline_rows))
display(ablation)

models = {
    'Logistic Regression - balanced': LogisticRegression(max_iter=500, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE),
    'Random Forest - balanced': RandomForestClassifier(n_estimators=80, max_depth=10, min_samples_leaf=50, class_weight='balanced_subsample', n_jobs=-1, random_state=RANDOM_STATE),
    'XGBoost - weighted': XGBClassifier(n_estimators=120, max_depth=4, learning_rate=0.08, subsample=0.85, colsample_bytree=0.85, eval_metric='logloss', scale_pos_weight=(model_train['is_click'].eq(0).sum()/model_train['is_click'].eq(1).sum()), random_state=RANDOM_STATE, n_jobs=-1, tree_method='hist')
}
results = baseline_rows.copy()
fitted = {}
probs_by_model = {}
for name, estimator in models.items():
    pipe, probs, rows = fit_eval_model(name, estimator, full_features, all_numeric_features)
    fitted[name] = pipe
    probs_by_model[name] = probs
    results.extend(rows)
metrics = pd.DataFrame(results).sort_values(['f1', 'roc_auc'], ascending=False)
display(metrics)

best_tuned = metrics[metrics['model'].str.endswith(' - tuned threshold')].iloc[0]
best_name = best_tuned['model'].replace(' - tuned threshold', '')
best_threshold = float(best_tuned['threshold'])
best_pipe = fitted[best_name]
best_probs = probs_by_model[best_name]
print('Selected:', best_name, 'threshold:', round(best_threshold, 3))
print(classification_report(y_valid, best_probs >= best_threshold, zero_division=0))
print('Selection rationale: highest F1 among evaluated tuned-threshold classifiers; PR-AUC and ROC-AUC leaders are reported separately.')

# Controlled SMOTENC experiment for the explicit business question.
# SMOTENC is applied before one-hot encoding, so categorical variables are not interpolated.
smote_numeric_features = all_numeric_features
smote_features = cat_features + smote_numeric_features
smote_train_idx, _ = train_test_split(
    np.arange(len(model_train)),
    train_size=min(50000, len(model_train)),
    stratify=model_train['is_click'],
    random_state=RANDOM_STATE,
)
smote_train = model_train.iloc[smote_train_idx].copy()
smote_y = smote_train['is_click']
smote_threshold_y = threshold_hist['is_click']
smote_holdout_y = holdout_hist['is_click']

smote_cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
])
smote_num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
smote_preprocess = ColumnTransformer([
    ('cat', smote_cat_transformer, cat_features),
    ('num', smote_num_transformer, smote_numeric_features),
])

start = time.perf_counter()
X_smote_base = smote_preprocess.fit_transform(smote_train[smote_features], smote_y)
X_smote_threshold = smote_preprocess.transform(threshold_hist[smote_features])
X_smote_holdout = smote_preprocess.transform(holdout_hist[smote_features])
cat_indices = list(range(len(cat_features)))

weighted_lr = LogisticRegression(max_iter=500, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE)
weighted_lr.fit(X_smote_base, smote_y)
weighted_threshold_scores = weighted_lr.predict_proba(X_smote_threshold)[:, 1]
weighted_threshold = best_threshold_by_f1(smote_threshold_y, weighted_threshold_scores)
weighted_holdout_scores = weighted_lr.predict_proba(X_smote_holdout)[:, 1]
weighted_time = time.perf_counter() - start
weighted_eval = evaluate_probs(smote_holdout_y, weighted_holdout_scores, 'Class weighting sample experiment', weighted_threshold)

start = time.perf_counter()
minority_count = int(smote_y.sum())
majority_count = int((smote_y == 0).sum())
target_minority = min(majority_count, minority_count * 3)
smotenc = SMOTENC(categorical_features=cat_indices, sampling_strategy={1: target_minority}, random_state=RANDOM_STATE, k_neighbors=5)
X_smote_resampled, y_smote_resampled = smotenc.fit_resample(X_smote_base, smote_y)
smotenc_lr = LogisticRegression(max_iter=500, n_jobs=-1, random_state=RANDOM_STATE)
smotenc_lr.fit(X_smote_resampled, y_smote_resampled)
smotenc_threshold_scores = smotenc_lr.predict_proba(X_smote_threshold)[:, 1]
smotenc_threshold = best_threshold_by_f1(smote_threshold_y, smotenc_threshold_scores)
smotenc_holdout_scores = smotenc_lr.predict_proba(X_smote_holdout)[:, 1]
smotenc_time = time.perf_counter() - start
smotenc_eval = evaluate_probs(smote_holdout_y, smotenc_holdout_scores, 'SMOTENC sample experiment', smotenc_threshold)

smotenc_comparison = pd.DataFrame([
    {
        'strategy': 'Class weighting',
        'training_rows_after_sampling': len(smote_train),
        'precision': weighted_eval['precision'],
        'recall': weighted_eval['recall'],
        'f1': weighted_eval['f1'],
        'pr_auc': weighted_eval['pr_auc'],
        'false_negatives': weighted_eval['fn'],
        'training_time_sec': weighted_time,
        'threshold': weighted_threshold,
    },
    {
        'strategy': 'SMOTENC',
        'training_rows_after_sampling': len(y_smote_resampled),
        'precision': smotenc_eval['precision'],
        'recall': smotenc_eval['recall'],
        'f1': smotenc_eval['f1'],
        'pr_auc': smotenc_eval['pr_auc'],
        'false_negatives': smotenc_eval['fn'],
        'training_time_sec': smotenc_time,
        'threshold': smotenc_threshold,
    }
])
base_fn = smotenc_comparison.loc[smotenc_comparison['strategy'].eq('Class weighting'), 'false_negatives'].iloc[0]
smote_fn = smotenc_comparison.loc[smotenc_comparison['strategy'].eq('SMOTENC'), 'false_negatives'].iloc[0]
base_rows = smotenc_comparison.loc[smotenc_comparison['strategy'].eq('Class weighting'), 'training_rows_after_sampling'].iloc[0]
smote_rows = smotenc_comparison.loc[smotenc_comparison['strategy'].eq('SMOTENC'), 'training_rows_after_sampling'].iloc[0]
smote_summary = {
    'fn_reduction_pct': float((base_fn - smote_fn) / base_fn * 100) if base_fn else 0,
    'training_rows_increase_pct': float((smote_rows - base_rows) / base_rows * 100),
    'conclusion': 'SMOTENC is preferred only if false-negative reduction justifies added training volume; otherwise class weighting is simpler.'
}
display(smotenc_comparison)
print(smote_summary)

# Business statistics
weekend = train_fe.groupby('is_weekend')['is_click'].agg(['mean','sum','count']).reset_index()
weekday_ctr = float(weekend.loc[weekend['is_weekend'].eq(0), 'mean'].iloc[0])
weekend_ctr = float(weekend.loc[weekend['is_weekend'].eq(1), 'mean'].iloc[0])
weekend_uplift_pp = (weekend_ctr - weekday_ctr) * 100
weekend_relative_uplift = (weekend_ctr / weekday_ctr - 1) * 100
z_stat, p_value = proportions_ztest(
    weekend.sort_values('is_weekend')['sum'].to_numpy(),
    weekend.sort_values('is_weekend')['count'].to_numpy()
)
print(f'Weekend CTR: {weekend_ctr:.2%}; Weekday CTR: {weekday_ctr:.2%}; uplift: {weekend_uplift_pp:.2f} pp / {weekend_relative_uplift:.1f}% relative; p-value={p_value:.4g}')

product = train_fe.groupby('product')['is_click'].agg(clicks='sum', impressions='count', ctr='mean').reset_index()
product['expected_clicks_per_100k_impressions'] = product['ctr'] * 100000
product = product.sort_values('ctr', ascending=False)
display(product)

profile = train_fe.groupby(['gender','age_level','city_development_index'])['is_click'].agg(clicks='sum', impressions='count', ctr='mean').reset_index()
profile = profile[profile['impressions'] >= 500].copy()
ci_low, ci_high = proportion_confint(profile['clicks'], profile['impressions'], method='wilson')
profile['ctr_ci_low'] = ci_low
profile['ctr_ci_high'] = ci_high
profile = profile.sort_values('ctr', ascending=False).head(10)
display(profile)

# Top individual feature importance; keep exact names instead of broad split('_')[0] grouping.
feature_names = best_pipe.named_steps['prep'].get_feature_names_out()
importances = getattr(best_pipe.named_steps['model'], 'feature_importances_', None)
if importances is not None:
    feature_importance = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False).head(15)
else:
    sample_n = min(15000, len(holdout_hist))
    sample = holdout_hist.sample(sample_n, random_state=RANDOM_STATE)
    perm = permutation_importance(best_pipe, sample[full_features], sample['is_click'], scoring='average_precision', n_repeats=3, random_state=RANDOM_STATE, n_jobs=-1)
    feature_importance = pd.DataFrame({'feature': full_features, 'importance': perm.importances_mean}).sort_values('importance', ascending=False).head(15)
display(feature_importance)

# Curves and charts
plt.figure(figsize=(6,4)); train['is_click'].value_counts().sort_index().plot(kind='bar', color=['#64748b','#f59e0b']); plt.xticks([0,1], ['No Click','Click'], rotation=0); plt.title('Target Distribution'); plt.ylabel('Impressions'); plt.tight_layout(); plt.savefig(IMAGE_DIR/'target_distribution.png', dpi=150); plt.show()

plt.figure(figsize=(8,4)); product.sort_values('ctr').plot(kind='barh', x='product', y='ctr', legend=False, ax=plt.gca(), color='#2f6f9f'); plt.title('Product CTR'); plt.xlabel('CTR'); plt.tight_layout(); plt.savefig(IMAGE_DIR/'product_ctr.png', dpi=150); plt.show()

plot_metrics = metrics[metrics['model'].str.endswith('tuned threshold')].sort_values('f1')
plt.figure(figsize=(8,4)); plot_metrics.plot(kind='barh', x='model', y='f1', legend=False, ax=plt.gca(), color='#f59e0b'); plt.title('Tuned-threshold F1 by Model'); plt.xlabel('F1'); plt.tight_layout(); plt.savefig(IMAGE_DIR/'model_f1.png', dpi=150); plt.show()

plt.figure(figsize=(8,4)); ablation.sort_values('f1').plot(kind='barh', x='model', y='f1', legend=False, ax=plt.gca(), color='#0f766e'); plt.title('Feature Ablation: F1 Lift'); plt.xlabel('F1'); plt.tight_layout(); plt.savefig(IMAGE_DIR/'feature_ablation.png', dpi=150); plt.show()

plt.figure(figsize=(8,4)); feature_importance.sort_values('importance').plot(kind='barh', x='feature', y='importance', legend=False, ax=plt.gca(), color='#14b8a6'); plt.title('Top Individual Feature Importances'); plt.tight_layout(); plt.savefig(IMAGE_DIR/'feature_importance.png', dpi=150); plt.show()

precision, recall, _ = precision_recall_curve(y_valid, best_probs)
plt.figure(figsize=(6,4)); plt.plot(recall, precision, color='#b91c1c', label='Model'); plt.axhline(y_valid.mean(), color='#64748b', linestyle='--', label=f'Prevalence baseline ({y_valid.mean():.2%})'); plt.title('Precision-Recall Curve'); plt.xlabel('Recall'); plt.ylabel('Precision'); plt.legend(); plt.tight_layout(); plt.savefig(IMAGE_DIR/'pr_curve.png', dpi=150); plt.show()

fpr, tpr, _ = roc_curve(y_valid, best_probs)
plt.figure(figsize=(6,4)); plt.plot(fpr, tpr, color='#2563eb'); plt.plot([0,1],[0,1],'--',color='#94a3b8'); plt.title('ROC Curve'); plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.tight_layout(); plt.savefig(IMAGE_DIR/'roc_curve.png', dpi=150); plt.show()

prob_true, prob_pred = calibration_curve(y_valid, best_probs, n_bins=10, strategy='quantile')
plt.figure(figsize=(6,4)); plt.plot(prob_pred, prob_true, marker='o', color='#7c3aed'); plt.plot([0,1],[0,1],'--',color='#94a3b8'); plt.title('Calibration Plot'); plt.xlabel('Mean predicted probability'); plt.ylabel('Observed click rate'); plt.tight_layout(); plt.savefig(IMAGE_DIR/'calibration_plot.png', dpi=150); plt.show()


# Decision-oriented evaluation: lift/gains, Precision@K/Recall@K, cost thresholds, CIs, data quality, and cold start.

def make_lift_table(y_true, scores, n_bins=10):
    frame = pd.DataFrame({'actual': np.asarray(y_true), 'score': np.asarray(scores)})
    frame = frame.sort_values('score', ascending=False).reset_index(drop=True)
    frame['decile'] = pd.qcut(frame.index + 1, q=n_bins, labels=[f'Top {i*10}-{(i+1)*10}%' for i in range(n_bins)], duplicates='drop')
    baseline_ctr = frame['actual'].mean()
    lift = frame.groupby('decile', observed=True).agg(
        impressions=('actual','size'), clicks=('actual','sum'), actual_ctr=('actual','mean'), avg_score=('score','mean')
    ).reset_index()
    lift['lift_vs_baseline'] = lift['actual_ctr'] / baseline_ctr
    lift['clicks_captured_pct'] = lift['clicks'] / frame['actual'].sum()
    lift['cumulative_clicks_captured_pct'] = lift['clicks'].cumsum() / frame['actual'].sum()
    return lift

def precision_recall_at_k(y_true, scores, ks=(0.01, 0.05, 0.10, 0.20)):
    frame = pd.DataFrame({'actual': np.asarray(y_true), 'score': np.asarray(scores)}).sort_values('score', ascending=False)
    total_clicks = frame['actual'].sum()
    baseline = frame['actual'].mean()
    rows = []
    for k in ks:
        top = frame.head(max(1, int(len(frame) * k)))
        precision_k = top['actual'].mean()
        rows.append({
            'targeted_impressions': f'Top {int(k*100)}%',
            'impressions_served': len(top),
            'precision_at_k': precision_k,
            'recall_at_k': top['actual'].sum() / total_clicks,
            'lift': precision_k / baseline,
        })
    return pd.DataFrame(rows)

def business_value_threshold_table(y_true, scores, scenarios):
    """Scenario-based ad economics for threshold selection.
    Values are assumptions because the dataset has no CPC, CPM, margin, or revenue fields.
    Utility = true_clicks_served * value_per_click - served_impressions * cost_per_impression.
    """
    thresholds = np.unique(np.round(np.linspace(0.01, 0.99, 199), 3))
    rows = []
    y_arr = np.asarray(y_true)
    total_clicks = y_arr.sum()
    for scenario, value_per_click, cost_per_impression in scenarios:
        best = None
        for threshold in thresholds:
            pred = (scores >= threshold).astype(int)
            tn, fp, fn, tp = confusion_matrix(y_arr, pred).ravel()
            served = tp + fp
            utility = tp * value_per_click - served * cost_per_impression
            row = {
                'scenario': scenario,
                'value_per_click': value_per_click,
                'cost_per_impression': cost_per_impression,
                'optimal_threshold': threshold,
                'served_impressions': int(served),
                'clicks_captured': int(tp),
                'clicks_missed': int(fn),
                'precision': precision_score(y_arr, pred, zero_division=0),
                'recall': recall_score(y_arr, pred, zero_division=0),
                'expected_utility': utility,
                'utility_per_1k_impressions': utility / len(y_arr) * 1000,
                'clicks_captured_pct': tp / total_clicks if total_clicks else 0,
            }
            if best is None or utility > best['expected_utility']:
                best = row
        rows.append(best)
    return pd.DataFrame(rows)

def bootstrap_metric_ci(y_true, model_score_pairs, n_boot=200):
    rng = np.random.default_rng(RANDOM_STATE)
    y_arr = np.asarray(y_true)
    rows = []
    for name, scores, threshold in model_score_pairs:
        scores = np.asarray(scores)
        f1_values, pr_values = [], []
        for _ in range(n_boot):
            idx = rng.integers(0, len(y_arr), len(y_arr))
            if len(np.unique(y_arr[idx])) < 2:
                continue
            f1_values.append(f1_score(y_arr[idx], scores[idx] >= threshold, zero_division=0))
            pr_values.append(average_precision_score(y_arr[idx], scores[idx]))
        rows.append({
            'model': name,
            'f1': f1_score(y_arr, scores >= threshold, zero_division=0),
            'f1_ci_low': np.percentile(f1_values, 2.5),
            'f1_ci_high': np.percentile(f1_values, 97.5),
            'pr_auc': average_precision_score(y_arr, scores),
            'pr_auc_ci_low': np.percentile(pr_values, 2.5),
            'pr_auc_ci_high': np.percentile(pr_values, 97.5),
        })
    return pd.DataFrame(rows)

def cold_start_table(frame, scores, y_true):
    out = frame.copy()
    out['score'] = np.asarray(scores)
    out['actual'] = np.asarray(y_true)
    rows = []
    for feature in ['user_product', 'campaign_product', 'webpage_product']:
        known = out[f'{feature}_hist_impressions'] > 0
        for label, mask in [('known_history', known), ('cold_start', ~known)]:
            subset = out[mask]
            rows.append({
                'feature': feature,
                'segment': label,
                'impressions': len(subset),
                'share_of_holdout': len(subset) / len(out),
                'ctr': subset['actual'].mean(),
                'roc_auc': roc_auc_score(subset['actual'], subset['score']) if subset['actual'].nunique() == 2 else np.nan,
                'pr_auc': average_precision_score(subset['actual'], subset['score']) if subset['actual'].nunique() == 2 else np.nan,
            })
    return pd.DataFrame(rows)

data_quality = pd.DataFrame([
    ['Training observations', len(train)],
    ['Test observations', len(test)],
    ['Clicks', int(train['is_click'].sum())],
    ['Non-clicks', int((train['is_click'] == 0).sum())],
    ['Click rate', f"{train['is_click'].mean():.2%}"],
    ['Duplicate rows', int(train.duplicated().sum())],
    ['Total missing cells', int(train.isna().sum().sum())],
    ['Unique products', train['product'].nunique()],
    ['Unique campaigns', train['campaign_id'].nunique()],
    ['Unique webpages', train['webpage_id'].nunique()],
    ['Date range', f"{train['DateTime'].min()} to {train['DateTime'].max()}"],
], columns=['Data check', 'Result'])
missing_summary = (train.isna().mean().mul(100).sort_values(ascending=False).reset_index().rename(columns={'index':'field', 0:'missing_pct'}).query('missing_pct > 0'))

decile_lift = make_lift_table(y_valid, best_probs)
precision_k = precision_recall_at_k(y_valid, best_probs)
business_value_scenarios = business_value_threshold_table(y_valid, best_probs, [
    ('Low click value / cheap media', 1.00, 0.05),
    ('Base ad economics', 2.00, 0.05),
    ('High click value', 5.00, 0.05),
    ('Expensive media', 2.00, 0.15),
])
model_score_pairs = []
for model_name, scores in probs_by_model.items():
    threshold = float(metrics.loc[metrics['model'].eq(model_name + ' - tuned threshold'), 'threshold'].iloc[0])
    model_score_pairs.append((model_name, scores, threshold))
bootstrap_ci = bootstrap_metric_ci(y_valid, model_score_pairs)
cold_start = cold_start_table(holdout_hist, best_probs, y_valid)

display(data_quality)
display(missing_summary)
display(decile_lift)
display(precision_k)
display(business_value_scenarios)
display(bootstrap_ci)
display(cold_start)

# Production lifecycle diagram.
fig, ax = plt.subplots(figsize=(10, 5))
ax.axis('off')
steps = ['Historical impressions', 'Feature engineering', 'Historical feature store', 'XGBoost click-propensity model', 'Ranking / bidding', 'Ad serving', 'Clicks + outcomes', 'Monitoring + retraining']
xs = np.linspace(0.07, 0.93, len(steps))
for i, (x, step) in enumerate(zip(xs, steps)):
    ax.text(x, 0.55, step, ha='center', va='center', fontsize=9, bbox=dict(boxstyle='round,pad=0.35', facecolor='#e0f2fe', edgecolor='#0284c7'))
    if i < len(steps) - 1:
        ax.annotate('', xy=(xs[i+1]-0.045, 0.55), xytext=(x+0.045, 0.55), arrowprops=dict(arrowstyle='->', color='#334155', lw=1.5))
ax.text(0.5, 0.22, 'Monitor: PR-AUC ? Lift@K ? CTR ? calibration ? feature drift ? latency', ha='center', fontsize=10, color='#475569')
ax.set_title('Production Lifecycle for CTR Scoring', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig(IMAGE_DIR/'production_lifecycle.png', dpi=150); plt.show()

business_mapping = pd.DataFrame([
    ['High product CTR', 'Increase exposure and reserve more inventory'],
    ['High webpage CTR', 'Prioritize placement and monitor page quality'],
    ['High user-product historical CTR', 'Personalize ad selection and test targeted bid uplift'],
    ['Weekend uplift', 'Test weekend bid modifier before permanent adoption'],
    ['High predicted click propensity', 'Prioritize impression under budget constraints'],
    ['Low CTR product', 'Refresh creative or reduce allocation'],
    ['Model drift', 'Retrain and recalibrate thresholds'],
], columns=['Model insight', 'Business action'])
display(business_mapping)

# Final train/test feature construction: training rows are cumulative; test sees all train history.
full_train_hist, full_prior = add_cumulative_history_features(train_fe, history_cols)
full_maps, _ = build_history_maps(train_fe, history_cols)
test_final = apply_history_maps(test_fe, full_maps, full_prior, history_cols)
final_pipe = Pipeline([('prep', make_preprocess(all_numeric_features)), ('model', models[best_name])])
# Final fit freezes the selected architecture and threshold, then uses all eligible
# historical training rows before scoring the test data. This keeps the threshold
# selection independent while using maximum available history for final scoring.
final_pipe.fit(full_train_hist[full_features], full_train_hist['is_click'])
test_pred = final_pipe.predict_proba(test_final[full_features])[:, 1]
submission = pd.DataFrame({
    'session_id': test['session_id'],
    'click_propensity_score': test_pred,
    'predicted_click': (test_pred >= best_threshold).astype(int),
})
submission.to_csv(OUTPUT_DIR/'ad_click_prediction_test_predictions.csv', index=False)
print(submission.head())

artifacts = {
    'train_shape': train.shape,
    'test_shape': test.shape,
    'click_rate': float(train['is_click'].mean()),
    'training_period_end_before': str(threshold_cutoff.date()),
    'threshold_tuning_date': str(threshold_cutoff.date()),
    'final_holdout_date': str(holdout_cutoff.date()),
    'metrics': metrics.to_dict('records'),
    'ablation': ablation.to_dict('records'),
    'best_model': best_name,
    'best_threshold': best_threshold,
    'weekend': weekend.to_dict('records'),
    'weekend_uplift_pp': weekend_uplift_pp,
    'weekend_relative_uplift_pct': weekend_relative_uplift,
    'weekend_p_value': float(p_value),
    'product': product.to_dict('records'),
    'profile': profile.to_dict('records'),
    'feature_importance': feature_importance.to_dict('records'),
    'business_mapping': business_mapping.to_dict('records'),
    'data_quality': data_quality.to_dict('records'),
    'missing_summary': missing_summary.to_dict('records'),
    'decile_lift': decile_lift.to_dict('records'),
    'precision_at_k': precision_k.to_dict('records'),
    'business_value_thresholds': business_value_scenarios.to_dict('records'),
    'smotenc_comparison': smotenc_comparison.to_dict('records'),
    'smote_summary': smote_summary,
    'bootstrap_ci': bootstrap_ci.to_dict('records'),
    'cold_start': cold_start.to_dict('records'),
    'performance_note': 'Moderate discrimination and imperfect calibration; use scores as click-propensity rankings until calibrated.',
    'final_training_note': 'The selected architecture and threshold are frozen after temporal validation; the final scoring model is retrained on all available labeled history before predicting the test set.'
}
(OUTPUT_DIR/'ctr_artifacts.json').write_text(json.dumps(artifacts, indent=2, default=str))
print('Artifacts written to outputs/ and assets/images/.')


Train: (463291, 15) Test: (128858, 14)
Click rate: 6.76 %


Training period ends before: 2017-07-06
Threshold-tuning date: 2017-07-06
Final holdout date: 2017-07-07
Train: (314299, 24) Threshold: (77526, 24) Holdout: (71466, 24) Holdout click rate: 0.0616
Random stratified split is not primary because future ad-serving behavior should not leak backward into training.


Model training sample: (150000, 57) Threshold tuning: (77526, 57) Final holdout: (71466, 57)


,model,threshold,roc_auc,pr_auc,f1,precision,recall,tn,fp,fn,tp
0,Global CTR probability baseline,0.070373,0.5,0.061582,0.116019,0.061582,1.0,0,67065,0,4401
1,Always non-click baseline,0.500000,0.5,0.061582,0.000000,0.000000,0.0,67065,0,4401,0


,model,threshold,roc_auc,pr_auc,f1,precision,recall,tn,fp,fn,tp
0,Baseline raw + temporal - tuned threshold,0.480,0.530088,0.067627,0.119275,0.065986,0.619859,28451,38614,1673,2728
1,+ Interaction history counts - tuned threshold,0.515,0.575911,0.076237,0.133436,0.078185,0.454897,43461,23604,2399,2002
2,+ Leakage-safe historical CTR - tuned threshold,0.465,0.595476,0.081703,0.139458,0.083119,0.432856,46051,21014,2496,1905


...

,scenario,value_per_click,cost_per_impression,optimal_threshold,served_impressions,clicks_captured,clicks_missed,precision,recall,expected_utility,utility_per_1k_impressions,clicks_captured_pct
0,Low click value / cheap media,1.0,0.05,0.391,50234,3557,844,0.070809,0.808225,1045.30,14.626536,0.808225
1,Base ad economics,2.0,0.05,0.183,70241,4381,20,0.062371,0.995456,5249.95,73.460807,0.995456
2,High click value,5.0,0.05,0.134,70872,4396,5,0.062027,0.998864,18436.40,257.974421,0.998864
3,Expensive media,2.0,0.15,0.490,20842,1804,2597,0.086556,0.409907,481.70,6.740268,0.409907


,model,f1,f1_ci_low,f1_ci_high,pr_auc,pr_auc_ci_low,pr_auc_ci_high
0,Logistic Regression - balanced,0.139458,0.134483,0.143917,0.081703,0.078473,0.085312
1,Random Forest - balanced,0.134357,0.127383,0.140196,0.081037,0.077254,0.084712
2,XGBoost - weighted,0.141692,0.135195,0.147860,0.081917,0.078106,0.085689


,feature,segment,impressions,share_of_holdout,ctr,roc_auc,pr_auc
0,user_product,known_history,35923,0.502659,0.049523,0.590068,0.067329
1,user_product,cold_start,35543,0.497341,0.073770,0.562495,0.088158
2,campaign_product,known_history,71114,0.995075,0.061549,0.593749,0.081914
3,campaign_product,cold_start,352,0.004925,0.068182,0.544715,0.094895
4,webpage_product,known_history,71114,0.995075,0.061549,0.593749,0.081914
5,webpage_product,cold_start,352,0.004925,0.068182,0.544715,0.094895


,Model insight,Business action
0,High product CTR,Increase exposure and reserve more inventory
1,High webpage CTR,Prioritize placement and monitor page quality
2,High user-product historical CTR,Personalize ad selection and test targeted bid...
3,Weekend uplift,Test weekend bid modifier before permanent ado...
4,High predicted click propensity,Prioritize impression under budget constraints
5,Low CTR product,Refresh creative or reduce allocation
6,Model drift,Retrain and recalibrate thresholds


   session_id  click_propensity_score  predicted_click
0      411705                0.595968                1
1      208263                0.271248                0
2      239450                0.272625                0
3      547761                0.332275                0
4      574275                0.571721                1
Artifacts written to outputs/ and assets/images/.


## 8. Limitations and Future Improvements

- Validation metrics show moderate discrimination, not high accuracy.
- Add more behavioral history, creative metadata, bid price, placement quality, and recency features if available.
- Use SMOTENC only with a categorical-aware implementation before one-hot encoding.
- Add probability calibration and live A/B testing before production deployment.
- Monitor drift, segment stability, fairness/privacy constraints, and threshold performance over time.